# Pipeline C — Step 2: XGBoost Pseudo-MIDAS Model

**Input:** `pipeline_C/output/C1_monthly_features.csv`

**Output:** Metrics, feature importance, comparison with Pipeline A

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def find_root(s=Path.cwd()):
    for p in [s.resolve(), *s.resolve().parents]:
        if (p/'model_panel.csv').exists(): return p
    raise FileNotFoundError

ROOT = find_root()
OUT  = ROOT / 'pipeline_C' / 'output'
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(OUT / 'C1_monthly_features.csv')
with open(OUT / 'C1_feature_meta.json') as f:
    meta = json.load(f)
target = meta['target']
feature_cols = meta['feature_cols']
print(f'Loaded {len(df)} rows, {len(feature_cols)} features')

Loaded 1116 rows, 32 features


## 1. Train/Test Split (Time-based)

In [2]:
# Temporal split: Train = 2023-2024, Test = 2025
train = df[df['tahun'].isin([2023, 2024])].copy()
test  = df[df['tahun'] == 2025].copy()

X_train, y_train = train[feature_cols], train[target]
X_test,  y_test  = test[feature_cols],  test[target]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

Train: (744, 32)  |  Test: (372, 32)


## 2. XGBoost Model

In [3]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=5,
    random_state=42,
    enable_categorical=False,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)
print('Model trained.')

Model trained.


## 3. Evaluation — Monthly Level

In [4]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

# Naive baseline: y_hat = y_{t-1}
y_naive = test['twp90_lag_1'].values

metrics = {
    'XGBoost (Train)': {
        'n': len(y_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'mae':  mean_absolute_error(y_train, y_pred_train),
        'r2':   r2_score(y_train, y_pred_train),
    },
    'XGBoost (Test)': {
        'n': len(y_test),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'mae':  mean_absolute_error(y_test, y_pred_test),
        'r2':   r2_score(y_test, y_pred_test),
    },
    'Naive y=y_{t-1} (Test)': {
        'n': len(y_test),
        'rmse': np.sqrt(mean_squared_error(y_test, y_naive)),
        'mae':  mean_absolute_error(y_test, y_naive),
        'r2':   r2_score(y_test, y_naive),
    },
}

df_metrics = pd.DataFrame(metrics).T
df_metrics['n'] = df_metrics['n'].astype(int)
print('=== MONTHLY-LEVEL METRICS ===')
print(df_metrics.to_string())

=== MONTHLY-LEVEL METRICS ===
                          n      rmse       mae        r2
XGBoost (Train)         744  0.001846  0.000931  0.968200
XGBoost (Test)          372  0.005437  0.001620  0.746826
Naive y=y_{t-1} (Test)  372  0.004923  0.002012  0.792422


## 4. Evaluation — Annual Aggregation (Comparable to Pipeline A)

In [5]:
# Aggregate monthly predictions to annual averages for fair comparison with Pipeline A
test_eval = test[['provinsi_id', 'nama_provinsi', 'tahun', target]].copy()
test_eval['pred'] = y_pred_test
test_eval['naive'] = y_naive

annual = test_eval.groupby(['provinsi_id', 'nama_provinsi']).agg(
    twp90_actual=('twp90_pct', 'mean'),
    twp90_pred=('pred', 'mean'),
    twp90_naive=('naive', 'mean'),
).reset_index()

# Pipeline A metrics (from A2_metrics.csv)
pA_rmse_test  = 0.005542
pA_r2_test    = 0.5999
pA_naive_rmse = 0.004768

# Pipeline C annual metrics
pC_rmse = np.sqrt(mean_squared_error(annual['twp90_actual'], annual['twp90_pred']))
pC_mae  = mean_absolute_error(annual['twp90_actual'], annual['twp90_pred'])
pC_r2   = r2_score(annual['twp90_actual'], annual['twp90_pred'])
pC_naive_rmse = np.sqrt(mean_squared_error(annual['twp90_actual'], annual['twp90_naive']))

print('='*60)
print('ANNUAL-LEVEL COMPARISON (Pipeline A vs Pipeline C)')
print('='*60)
print(f'{"Metric":<25} {"Pipeline A (Huber FD)":>20} {"Pipeline C (XGBoost)":>20}')
print('-'*65)
print(f'{"Test RMSE":<25} {pA_rmse_test:>20.6f} {pC_rmse:>20.6f}')
print(f'{"Test R²":<25} {pA_r2_test:>20.4f} {pC_r2:>20.4f}')
print(f'{"Naive RMSE":<25} {pA_naive_rmse:>20.6f} {pC_naive_rmse:>20.6f}')
print(f'{"Beats Naive?":<25} {"NO":>20} {"YES" if pC_rmse < pC_naive_rmse else "NO":>20}')
print(f'{"RMSE Improvement vs A":<25} {"---":>20} {(1 - pC_rmse/pA_rmse_test)*100:>19.1f}%')

# Save comparison
comp = pd.DataFrame([
    {'pipeline': 'A (Huber FD)', 'level': 'annual', 'rmse': pA_rmse_test, 'r2': pA_r2_test, 'naive_rmse': pA_naive_rmse},
    {'pipeline': 'C (XGBoost)',  'level': 'annual', 'rmse': pC_rmse,      'r2': pC_r2,      'naive_rmse': pC_naive_rmse},
])
comp.to_csv(OUT / 'C2_pipeline_comparison.csv', index=False)

ANNUAL-LEVEL COMPARISON (Pipeline A vs Pipeline C)
Metric                    Pipeline A (Huber FD) Pipeline C (XGBoost)
-----------------------------------------------------------------
Test RMSE                             0.005542             0.002531
Test R²                                 0.5999               0.9166
Naive RMSE                            0.004768             0.001380
Beats Naive?                                NO                   NO
RMSE Improvement vs A                      ---                54.3%


## 5. Feature Importance

In [6]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importances.head(20).sort_values().plot.barh(ax=ax, color='#3b82f6')
ax.set_title('Pipeline C — XGBoost Feature Importance (Top 20)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance (Gain)')
plt.tight_layout()
fig.savefig(OUT / 'C2_feature_importance.png', dpi=150)
plt.show()
print('\nTop 10 features:')
print(importances.head(10).to_string())


Top 10 features:
twp90_roll3_mean        0.546706
twp90_lag_1             0.318735
twp90_lag_6             0.016702
twp90_mom               0.013169
x6_tabungan_miliar      0.009906
x1_bi_rate_pct          0.007568
x9_npl_ratio            0.006879
twp90_lag_2             0.006550
x9_npl_ratio_x_month    0.006532
twp90_lag_3             0.006234


## 6. Per-Province Comparison

In [7]:
annual['error'] = annual['twp90_actual'] - annual['twp90_pred']
annual['abs_error'] = annual['error'].abs()
annual = annual.sort_values('twp90_actual', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
x = range(len(annual))
ax.bar(x, annual['twp90_actual'], width=0.4, label='Actual 2025', color='#3b82f6', alpha=0.8)
ax.bar([i+0.4 for i in x], annual['twp90_pred'], width=0.4, label='XGBoost Pred', color='#f59e0b', alpha=0.8)
ax.set_xticks([i+0.2 for i in x])
ax.set_xticklabels(annual['nama_provinsi'], rotation=75, ha='right', fontsize=8)
ax.set_ylabel('TWP90 (Annual Avg)')
ax.set_title('Pipeline C — Actual vs Predicted TWP90 per Province (2025)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(OUT / 'C2_per_province.png', dpi=150)
plt.show()

annual.to_csv(OUT / 'C2_annual_predictions.csv', index=False)
print(f'Saved per-province predictions.')

Saved per-province predictions.


## 7. Save All Metrics

In [8]:
df_metrics.to_csv(OUT / 'C2_monthly_metrics.csv')

# Summary text file
summary = f"""Pipeline C — XGBoost Pseudo-MIDAS Results
{'='*55}
Model        : XGBRegressor (n_est=300, depth=5, lr=0.05)
Train data   : Monthly 2023-2024 ({len(X_train)} obs)
Test data    : Monthly 2025 ({len(X_test)} obs)
Features     : {len(feature_cols)} total

MONTHLY METRICS
{'-'*40}
Train RMSE   : {np.sqrt(mean_squared_error(y_train, y_pred_train)):.6f}
Test  RMSE   : {np.sqrt(mean_squared_error(y_test, y_pred_test)):.6f}
Test  R2     : {r2_score(y_test, y_pred_test):.4f}
Naive RMSE   : {np.sqrt(mean_squared_error(y_test, y_naive)):.6f}

ANNUAL COMPARISON vs PIPELINE A
{'-'*40}
Pipeline A RMSE : {pA_rmse_test:.6f} (Huber FD)
Pipeline C RMSE : {pC_rmse:.6f} (XGBoost)
Pipeline C R2   : {pC_r2:.4f}
Improvement     : {(1 - pC_rmse/pA_rmse_test)*100:.1f}%
"""
with open(OUT / 'C2_summary.txt', 'w') as f:
    f.write(summary)
print(summary)

Pipeline C — XGBoost Pseudo-MIDAS Results
Model        : XGBRegressor (n_est=300, depth=5, lr=0.05)
Train data   : Monthly 2023-2024 (744 obs)
Test data    : Monthly 2025 (372 obs)
Features     : 32 total

MONTHLY METRICS
----------------------------------------
Train RMSE   : 0.001846
Test  RMSE   : 0.005437
Test  R2     : 0.7468
Naive RMSE   : 0.004923

ANNUAL COMPARISON vs PIPELINE A
----------------------------------------
Pipeline A RMSE : 0.005542 (Huber FD)
Pipeline C RMSE : 0.002531 (XGBoost)
Pipeline C R2   : 0.9166
Improvement     : 54.3%

